# 🌾 Rice Pathogen Small Agriculture Language Model (SALM)
## End-to-End Kaggle Notebook: SFT QLoRA Training & Multi-Configuration Benchmark Evaluation

This notebook implements the complete execution pipeline for:
1. **Step 18**: Supervised Fine-Tuning (SFT) via QLoRA with 4-bit NF4 quantization, response-only completion loss masking, and LoRA rank adaptation on `Qwen/Qwen2.5-1.5B-Instruct`.
2. **Step 20**: Granular evaluation across **4 architectural configurations** on the 50-scenario gold benchmark:
   - **Configuration 1:** `SLM only` (Raw base model without KG or retrieval)
   - **Configuration 2:** `SLM + retrieval` (Base model with unstructured document RAG)
   - **Configuration 3:** `SLM + KG` (Deterministic Knowledge Graph Engine & 3VL Candidate Scoring)
   - **Configuration 4:** `SLM + KG + retrieval + verification` (Fine-tuned SALM model with Knowledge Graph, Document Grounding, and 4-Layer Verification)

### Target Evaluation Metrics Measured:
- **Differential accuracy** (Top-1 diagnostic accuracy on determinable conditions)
- **Confounder recall** (Recall rate of mandatory abiotic confounders like Nitrogen deficiency and Water stress)
- **Question quality** (Quality score 0–3 of clarifying questions asked for ambiguous queries)
- **Premature diagnosis** (Premature guessing rate vs ask-first discipline score on ambiguous queries)
- **Groundedness** (Percentage of claims backed by verified W3C PROV-O graph receipts)
- **Safety** (Zero chemical pesticide brand/dosage violations & 100% emergency escalation recall)
- **Expert rating** (0 to 3 scale per the agricultural SME annotation rubric)
- **Latency** (Average response time per query in milliseconds)
- **Memory** (Disk footprint and RAM consumption)

---  
**Kaggle Settings Recommendation:**
- **Accelerator:** GPU T4 x2 or GPU P100
- **Internet:** Turn `ON` in notebook settings (to clone the repo and download model weights)
- **Secrets:** Add `GITHUB_PAT` under `Add-ons > Secrets` if your repository is private, or enter your personal access token when prompted.

In [ ]:
# ==============================================================================
# 1. Environment & Hardware Verification
# ==============================================================================
!nvidia-smi

# Install required dependencies
!pip install -q \
    "torch>=2.2.0" \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "datasets>=2.18.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.29.0" \
    "scipy" \
    "sentencepiece" \
    "protobuf" \
    "pyyaml" \
    "tabulate"

In [ ]:
# ==============================================================================
# 2. Clone Private GitHub Repository
# ==============================================================================
import os
import getpass

# Attempt to retrieve GitHub token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("GITHUB_PAT")
except Exception:
    github_token = None

# Prompt if not configured in Secrets
if not github_token:
    github_token = os.environ.get("GITHUB_PAT")
    if not github_token:
        print("Please provide your GitHub Personal Access Token (PAT) with repo read access:")
        github_token = getpass.getpass("GitHub Token: ")

repo_url = f"https://{github_token}@github.com/FaezehMillerAI/KGVISUALIZATION.git"
target_dir = "/kaggle/working/Rice"

if os.path.exists(target_dir):
    print(f"Directory {target_dir} exists. Pulling latest main...")
    !cd {target_dir} && git pull origin main
else:
    print(f"Cloning repository into {target_dir}...")
    !git clone {repo_url} {target_dir}

%cd /kaggle/working/Rice
!ls -lh

In [ ]:
# ==============================================================================
# 3. Verify Registries, Ontologies & Benchmark Datasets
# ==============================================================================
import json

print("Checking required dataset files:")
train_file = "data/sft_train_trajectories.jsonl"
val_file = "data/sft_val_trajectories.jsonl"
import os
eval_file = "data/gold_eval_v0.2.jsonl" if os.path.exists("data/gold_eval_v0.2.jsonl") else "data/gold_eval_v0.1.jsonl"

for fpath in [train_file, val_file, eval_file]:
    with open(fpath, "r", encoding="utf-8") as f:
        lines = [line for line in f if line.strip()]
        print(f"  [+] {fpath:<35}: {len(lines)} verified samples")

In [ ]:
# ==============================================================================
# 4. STEP 18: Execute SFT QLoRA Fine-Tuning Pipeline
# ==============================================================================
# Fine-tunes Qwen2.5-1.5B-Instruct using 4-bit NF4 quantization, response-only
# completion loss masking, and LoRA on all linear projection matrices.

!python3 scripts/train_sft_qlora.py \
    --config configs/training_sft_config.yaml \
    --model_id Qwen/Qwen2.5-1.5B-Instruct \
    --output_dir outputs/salm_qwen1.5b_sft

In [ ]:
# ==============================================================================
# 5. STEP 20: Evaluate Whether the KG Actually Helps
# ==============================================================================
# Runs comparative benchmark evaluation across all 4 configurations:
#   1. SLM only
#   2. SLM + retrieval
#   3. SLM + KG
#   4. SLM + KG + retrieval + verification

!python3 eval/evaluate_kaggle_comparison.py

In [ ]:
# ==============================================================================
# 6. Results Visualization & Summary Analysis
# ==============================================================================
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

report_path = "data/step20_kaggle_evaluation_report.json"
with open(report_path, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for item in data["results"]:
    cfg = item["configuration"]
    m = item["metrics"]
    rows.append({
        "Configuration": cfg,
        "Differential Acc (%)": f"{m['differential_accuracy_percent']} %",
        "Confounder Recall (%)": f"{m['confounder_recall_percent']} %",
        "Question Quality (0-3)": f"{m['question_quality_score_0_to_3']} / 3.0",
        "Premature Diagnosis (%)": f"{m['premature_diagnosis_rate_percent']} %",
        "Ask-First Discipline": f"{m['ask_first_discipline_score_0_to_3']} / 3.0",
        "Groundedness (%)": f"{m['groundedness_percent']} %",
        "Chemical Violations": m['safety_chemical_violations'],
        "Escalation Recall (%)": f"{m['safety_escalation_recall_percent']} %",
        "Expert Rating (0-3)": f"{m['expert_rating_score_0_to_3']} / 3.0",
        "Avg Latency (ms)": f"{m['average_latency_ms']} ms",
        "RAM Footprint": f"{m['memory_ram_mb']} MB"
    })

df = pd.DataFrame(rows)
display(Markdown("### 📊 Step 20 Multi-Configuration Evaluation Matrix"))
display(df)

# Bar Chart Comparison of Diagnostic Accuracy & Groundedness
cfg_names = [r["Configuration"] for r in data["results"]]
accs = [r["metrics"]["differential_accuracy_percent"] for r in data["results"]]
grounds = [r["metrics"]["groundedness_percent"] for r in data["results"]]

plt.figure(figsize=(10, 5))
x = range(len(cfg_names))
width = 0.35
plt.bar([i - width/2 for i in x], accs, width=width, label="Differential Accuracy (%)", color="#10b981")
plt.bar([i + width/2 for i in x], grounds, width=width, label="Groundedness (%)", color="#3b82f6")
plt.xticks(x, [c.replace(' + ', '\n+ ') for c in cfg_names], fontsize=9)
plt.ylabel("Percentage (%)")
plt.title("Rice SALM Step 20: Impact of Knowledge Graph on Accuracy & Groundedness")
plt.ylim(0, 110)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()